In [67]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path



In [68]:
%pip install docx2txt


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [69]:
import sys
!{sys.executable} -m pip install docx2txt


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [70]:
import docx2txt
print("found")

found


In [71]:
def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)

    #find all files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents


def process_all_docx(docx_directory):
    all_documents = []
    docx_dir = Path(docx_directory)

    docx_files = list(docx_dir.glob("**/*.docx"))
    print(f"Found {len(docx_files)} Docx files to process")

    for file in docx_files:
        print(f"\nProcessing: {file.name}")
        try: 
            loader = Docx2txtLoader(str(file))
            documents = loader.load()

            for doc in documents: 
                doc.metadata['source_file'] = file.name
                doc.metadata['file_type'] = 'docx'

            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} documents")

        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

            




# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

all_docx_documents = process_all_docx("../data")

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)


Found 1 PDF files to process

Processing: anudeep_manda_openai_coverletter.pdf
  ✓ Loaded 1 pages

Total documents loaded: 1
Found 1 Docx files to process

Processing: anudeep_manda_amex.docx
  ✓ Loaded 1 documents

Total documents loaded: 1


In [72]:

all_pdf_documents

[Document(metadata={'producer': 'macOS Version 15.6 (Build 24G84) Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20260116012304Z00'00'", 'title': 'anudeep_dropbox_coverletter', 'author': 'Anudeep Manda', 'moddate': "D:20260116012304Z00'00'", 'source': '../data/pdf/anudeep_manda_openai_coverletter.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'anudeep_manda_openai_coverletter.pdf', 'file_type': 'pdf'}, page_content='Anudeep Manda manda1@purdue.edu | (925) 683-7968 | linkedin.com/in/anudeepmanda Dear OpenAI Applied Engineering Team, I am writing to express my strong interest in the Software Engineering Intern (Emerging Talent) role on the Applied Engineering team at OpenAI. I am a Computer Engineering and Applied Mathematics student at Purdue University, and I am deeply motivated by OpenAI’s mission to responsibly deploy AI systems that are useful, reliable, and accessible to millions of users worldwide. OpenAI’s emphasis on learning from real-world depl

In [73]:
def split_documents(document, chunk_size =700, chunk_overlap = 50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len, 
        separators=  ["\n\n", "\n", " ", "","•", "-"]
    )
    split_docs = text_splitter.split_documents(document)
    print(f"Split {len(document)} documents into {len(split_docs)} chunks")
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs


In [74]:
chunks_pdf = split_documents(all_pdf_documents)
chunks_docx = split_documents(all_docx_documents)
print(chunks_pdf)
print(chunks_docx)


Split 1 documents into 5 chunks

Example chunk:
Content: Anudeep Manda manda1@purdue.edu | (925) 683-7968 | linkedin.com/in/anudeepmanda Dear OpenAI Applied Engineering Team, I am writing to express my strong interest in the Software Engineering Intern (Eme...
Metadata: {'producer': 'macOS Version 15.6 (Build 24G84) Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20260116012304Z00'00'", 'title': 'anudeep_dropbox_coverletter', 'author': 'Anudeep Manda', 'moddate': "D:20260116012304Z00'00'", 'source': '../data/pdf/anudeep_manda_openai_coverletter.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'anudeep_manda_openai_coverletter.pdf', 'file_type': 'pdf'}
Split 1 documents into 8 chunks

Example chunk:
Content: Anudeep Manda    

manda1@purdue.edu | 925-683-7968 | linkedin.com/in/anudeepmanda   

EDUCATION

							Purdue University – West Lafayette, IN       	  	  	  	  	  	  	       GPA: 3.8   May 2027  

	...
Metadata: {'source': '../data/docx/anudeep_manda_

In [75]:
# Embedding and Vector Store

In [76]:
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [77]:
class EmbeddingManager:
    #handles domcument embedding gneration using SentenceTransfomer

    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        '''
        Initialie the embedding manager
        model_name -> HuggingFace model name for sentences embeddings

        '''
        self.model_name = model_name
        self.model = None
        self._load_model() #loads model 

    def _load_model(self): #protected funciton 
        try:
            print(f"Loading embdding model : {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded succesfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}") # every text converted into dimensions
        except Exception as e:
            print(f"Error loading model {self.model}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]):
        if not self.model: 
            raise ValueError("Model not loaded")
        print(f"Generate Embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar= True)
        print(f"Generate embeddings with shape: {embeddings.shape}")
        return embeddings


##initializze the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager

Loading embdding model : all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2262.64it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded succesfully. Embedding dimension: 384


In [78]:
#vector store

In [79]:
import numpy as np
import os

class VectorStore:
    def __init__(self, collection_name: str = 'pdf_documents', persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try: 
            #create persistent chromadb client 
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            #create or get collection 
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata = {"description": "PDF document embeddings for RAG"}

            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e: 
            print(f"Error intializing vector store: {e}")
            raise
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
            #add documents and their embeddings to the vector store 
            #args: documents: list of langchain documents, embeddings their corresponding embeddings for the documents
            
            if len(documents) != len(embeddings):
                raise ValueError("Number of documents must match with the number of embeddings")
            
            print(f"Addings {len(documents)} documents to the vector store")
            

            #data preperation for ChromaDB 
            ids= []
            metadata_list = []
            documents_text = []
            embeddings_list = []

            for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
                #genereate unique id
                doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
                ids.append(doc_id)

                #prepare metadata
                metadata = dict(doc.metadata)
                metadata['doc_index'] = i 
                metadata['content_length'] = len(doc.page_content)
                metadata_list.append(metadata)
                #documents content
                documents_text.append(doc.page_content)
                #embedding
                embeddings_list.append(embedding.tolist())


            #add to collection 
            try:
                self.collection.add(
                    ids = ids, 
                    embeddings = embeddings, 
                    metadatas = metadata_list,
                    documents = documents_text

                )
                print(f"Successfully added {len(documents)} chunks to the vector store")
                print(f"Total chunks in collection: {self.collection.count()}")


            except Exception as e: 
                print(f"Error adding chunks to vector store: {e}")
                raise

    def clear_collection(self):
        try: 
            self.client.delete_collection(name = self.collection_name)
            print("Successfully cleared the collection")
            self._initialize_store()

        except Exception as e: 
            print(f"Error clearing the collection : {e}")
            raise
vectorstore = VectorStore()
vectorstore




Vector store initialized. Collection: pdf_documents
Existing documents in collection: 10


In [80]:
#convert chunks texts in embeddings

texts_pdf = [doc.page_content for doc in chunks_pdf]
texts_docx = [doc.page_content for doc in chunks_docx]

#generate the embeddings

embeddings_pdf = embedding_manager.generate_embeddings(texts_pdf)
embeddings_docx = embedding_manager.generate_embeddings(texts_docx)

#store in the vector database
vectorstore.add_documents(chunks_pdf, embeddings_pdf)
vectorstore.add_documents(chunks_docx, embeddings_docx)

# how to remove duplicates?


Generate Embeddings for 5 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.44it/s]


Generate embeddings with shape: (5, 384)
Generate Embeddings for 8 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  7.70it/s]

Generate embeddings with shape: (8, 384)
Addings 5 documents to the vector store
Successfully added 5 chunks to the vector store
Total chunks in collection: 15
Addings 8 documents to the vector store
Successfully added 8 chunks to the vector store
Total chunks in collection: 23


In [81]:
chunks_pdf

[Document(metadata={'producer': 'macOS Version 15.6 (Build 24G84) Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20260116012304Z00'00'", 'title': 'anudeep_dropbox_coverletter', 'author': 'Anudeep Manda', 'moddate': "D:20260116012304Z00'00'", 'source': '../data/pdf/anudeep_manda_openai_coverletter.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'anudeep_manda_openai_coverletter.pdf', 'file_type': 'pdf'}, page_content='Anudeep Manda manda1@purdue.edu | (925) 683-7968 | linkedin.com/in/anudeepmanda Dear OpenAI Applied Engineering Team, I am writing to express my strong interest in the Software Engineering Intern (Emerging Talent) role on the Applied Engineering team at OpenAI. I am a Computer Engineering and Applied Mathematics student at Purdue University, and I am deeply motivated by OpenAI’s mission to responsibly deploy AI systems that are useful, reliable, and accessible to millions of users worldwide. OpenAI’s emphasis on learning from real-world depl

In [82]:
#retrieval pipeline from vectorstore

class RagRetriever:
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):     
        #vector_store containes the document embeddings
        #embeddings manager for generating query embeddings

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0):   
        #query -> search query
        #top_k results to return 
        #score_threshold minimum similarity score threshold

        print(f"Retrievering documtnes for query: {query}")
        print(f"Top K: {top_k}, Minimum Threshold: {score_threshold}")

        # Preprocess query (optional: lowercase, strip)
        query = query.strip().lower()
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results = top_k
            )
            retrieved_docs = []

            # Extract the relevant fields from results
            ids = results["ids"][0]
            documents = results["documents"][0]
            metadatas = results["metadatas"][0]
            distances = results["distances"][0]

            for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                similarity_score = 1 - distance
                # Debug print for each chunk
                print(f"Chunk {i}: score={similarity_score:.4f}, content={document[:100].replace('\n',' ')}")

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        'id': doc_id,
                        'metadata': metadata,
                        'content': document, 
                        'similarity_score' : similarity_score, 
                        'distance' : distance, 
                        'rank' : i + 1
                    })
                    print(f"Retrieved {len(retrieved_docs)} documents")
                else:
                    print(f"No doucments found for chunk {i}")
            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
        

rag_retriever=RagRetriever(vectorstore,embedding_manager)


In [83]:
rag_retriever

In [84]:
rag_retriever.retrieve('''Qualification
check
Represents the skills you have
Find out how your skills align with this job's requirements. If anything seems off, you can easily click on the tags to select or unselect skills to reflect your actual expertise.

checkMachine Learning
checkPython
checkC/C++
checkAlgorithms
checkData Structures
checkStatistics
checkCollaboration
checkProblem Solving
Required
Currently pursuing a BS/MS/PhD in EE/ECE/CE/CS
Possessing deep knowledge of math, probability, statistics, and algorithms
Experienced in solving problems with Machine Learning models
Skilled in algorithms, data structures, and software development using Python and C/C++
Ability to develop ML-based tools to improve PPA and turnaround time for chip implementation
Work closely with other PD engineers to develop ML tools for areas such as synthesis, PnR, timing closure, and power grid analysis
Responsibility for selecting appropriate datasets, data representation methods, and implementing new algorithms
Capability to run machine learning tests, perform statistical analysis, and fine-tune models using test resultss''')

Retrievering documtnes for query: Qualification
check
Represents the skills you have
Find out how your skills align with this job's requirements. If anything seems off, you can easily click on the tags to select or unselect skills to reflect your actual expertise.

checkMachine Learning
checkPython
checkC/C++
checkAlgorithms
checkData Structures
checkStatistics
checkCollaboration
checkProblem Solving
Required
Currently pursuing a BS/MS/PhD in EE/ECE/CE/CS
Possessing deep knowledge of math, probability, statistics, and algorithms
Experienced in solving problems with Machine Learning models
Skilled in algorithms, data structures, and software development using Python and C/C++
Ability to develop ML-based tools to improve PPA and turnaround time for chip implementation
Work closely with other PD engineers to develop ML tools for areas such as synthesis, PnR, timing closure, and power grid analysis
Responsibility for selecting appropriate datasets, data representation methods, and implemen

Batches: 100%|██████████| 1/1 [00:00<00:00, 131.85it/s]

Generate embeddings with shape: (1, 384)
Chunk 0: score=0.0023, content=Anudeep Manda      manda1@purdue.edu | 925-683-7968 | linkedin.com/in/anudeepmanda     EDUCATION  		
Retrieved 1 documents
Chunk 1: score=-0.0961, content=Anudeep Manda      manda1@purdue.edu | 925-683-7968 | linkedin.com/in/anudeepmanda     EDUCATION  		
No doucments found for chunk 1
Chunk 2: score=-0.1547, content=Operating Systems: Linux (Ubuntu), Unix; shell scripting, process & resource monitoring  Web & Softw
No doucments found for chunk 2
Chunk 3: score=-0.2713, content=EXPERIENCE    USG Manufacturing – Smart Manufacturing Company focused on real time automation and an
No doucments found for chunk 3
Chunk 4: score=-0.2737, content=TECHNICAL SKILLS  Languages: Python, Java, C++, JavaScript/TypeScript, SQL, MATLAB, Go  Backend & Cl
No doucments found for chunk 4


[{'id': 'doc_e3ec5f78_0',
  'metadata': {'source_file': 'anudeep_manda_amex.docx',
   'file_type': 'docx',
   'content_length': 640,
   'source': '../data/docx/anudeep_manda_amex.docx',
   'doc_index': 0},
  'content': 'Anudeep Manda    \n\nmanda1@purdue.edu | 925-683-7968 | linkedin.com/in/anudeepmanda   \n\nEDUCATION\n\n\t\t\t\t\t\t\tPurdue University – West Lafayette, IN       \t  \t  \t  \t  \t  \t  \t       GPA: 3.8   May 2027  \n\n\t\t\t\t\tB.S. in Computer Engineering and Applied Mathematics (Double Major)  \t     \t  \t  \t  \t               \n\nCoursework: Computer Architecture, Advanced C Programming, Object-Oriented Programming (C++), Data Structures & Algorithms, Embedded Systems, Microprocessor Systems & Interfacing, Signals & Systems\n\nAdditional Certifications: AWS SageMaker, Full-Stack Developer Path (Microsoft Learn), MIT 6.S081 (Operating Systems)\n\nTECHNICAL SKILLS',
  'similarity_score': 0.0023250579833984375,
  'distance': 0.9976749420166016,
  'rank': 1}]

In [88]:
rag_retriever.retrieve("I am writing to express my strong interest in the Software Engineering Intern Emerging Talent role on the Applied Engineering team at OpenAI. I am a Computer Engineering and Applied Mathematics stud")

Retrievering documtnes for query: I am writing to express my strong interest in the Software Engineering Intern Emerging Talent role on the Applied Engineering team at OpenAI. I am a Computer Engineering and Applied Mathematics stud
Top K: 5, Minimum Threshold: 0.0
Generate Embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.44it/s]

Generate embeddings with shape: (1, 384)
Chunk 0: score=0.4800, content=Anudeep Manda manda1@purdue.edu | (925) 683-7968 | linkedin.com/in/anudeepmanda Dear OpenAI Applied 
Retrieved 1 documents
Chunk 1: score=0.3707, content=Anudeep Manda manda1@purdue.edu | (925) 683-7968 | linkedin.com/in/anudeepmanda Dear OpenAI Applied 
Retrieved 2 documents
Chunk 2: score=0.3058, content=researchers, and product teams to build scalable, user-focused AI experiences that have a meaningful
Retrieved 3 documents
Chunk 3: score=0.2840, content=particularly excited about the opportunity to contribute to OpenAI’s efforts to bring AI safely and 
Retrieved 4 documents
Chunk 4: score=0.0362, content=structured, queryable formats for downstream analytics and AI workflows. I regularly profiled latenc
Retrieved 5 documents


[{'id': 'doc_f9195e6a_0',
  'metadata': {'page_label': '1',
   'source': '../data/pdf/anudeep_manda_openai_coverletter.pdf',
   'producer': 'macOS Version 15.6 (Build 24G84) Quartz PDFContext',
   'creator': 'Word',
   'creationdate': "D:20260116012304Z00'00'",
   'page': 0,
   'total_pages': 1,
   'content_length': 699,
   'file_type': 'pdf',
   'doc_index': 0,
   'title': 'anudeep_dropbox_coverletter',
   'moddate': "D:20260116012304Z00'00'",
   'author': 'Anudeep Manda',
   'source_file': 'anudeep_manda_openai_coverletter.pdf'},
  'content': 'Anudeep Manda manda1@purdue.edu | (925) 683-7968 | linkedin.com/in/anudeepmanda Dear OpenAI Applied Engineering Team, I am writing to express my strong interest in the Software Engineering Intern (Emerging Talent) role on the Applied Engineering team at OpenAI. I am a Computer Engineering and Applied Mathematics student at Purdue University, and I am deeply motivated by OpenAI’s mission to responsibly deploy AI systems that are useful, reliable